# ⚡ NEXUS Stock AI — Phases 2, 3, & 4
## Data Cleaning, 4:00 PM Market Cutoff Enforcement & Calendar Alignment

**Objective:** Prepare model-ready time-series datasets by cleaning price data (Phase 2), cleaning news timestamps into US Eastern Time (Phase 3), and rigorously aligning news publications with actual market trading sessions while preventing lookahead leakage (Phase 4).

### 🛡️ The 4:00 PM Market Cutoff & Leakage Prevention Rules
1. **Intraday Market Cutoff:** Regular US stock exchanges close at 16:00:00 US/Eastern.
   - Articles published **before or at 16:00:00 US/Eastern** affect the **same trading day**.
   - Articles published **after 16:00:00 US/Eastern** cannot affect today's close and must target the **next trading day**.
2. **Market Calendar Alignment (Vectorized):**
   - Weekends and market holidays (e.g. Good Friday, Memorial Day, Christmas) are non-trading days.
   - Using `pd.merge_asof(direction='forward')`, weekend/holiday targets automatically roll forward to the **next available valid market trading date** for that specific ticker.

### ⚙️ Step 1: Environment Setup & Google Drive Mount

In [5]:
# Mount Google Drive to load Phase 1 persistent artifacts
import os
import sys
import gc
import time
import shutil
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

try:
    from google.colab import drive
    print("Mounting Google Drive at /content/drive...")
    drive.mount("/content/drive")
    print("✓ Google Drive mounted successfully.")
except Exception as e:
    print(f"Drive mount note: {e}")

# Configure file paths
GDRIVE_DIR = "/content/drive/MyDrive/NEXUS_Stock_AI/data"
LOCAL_DIR = "./data"

def resolve_path(filename):
    gdrive_p = os.path.join(GDRIVE_DIR, filename)
    local_p = os.path.join(LOCAL_DIR, filename)
    if os.path.exists(gdrive_p):
        return gdrive_p
    elif os.path.exists(local_p):
        return local_p
    return gdrive_p

PRICES_INPUT = resolve_path("filtered_prices.parquet")
NEWS_INPUT = resolve_path("filtered_news.parquet")

print(f"Prices input: {PRICES_INPUT} (Exists: {os.path.exists(PRICES_INPUT)})")
print(f"News input:   {NEWS_INPUT} (Exists: {os.path.exists(NEWS_INPUT)})")

Mounting Google Drive at /content/drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted successfully.
Prices input: /content/drive/MyDrive/NEXUS_Stock_AI/data/filtered_prices.parquet (Exists: True)
News input:   /content/drive/MyDrive/NEXUS_Stock_AI/data/filtered_news.parquet (Exists: True)


### 📈 Step 2: Phase 2 — Clean Prices & Extract Trading Calendar
- Standardize columns (`date`, `stock_symbol`, `open`, `high`, `low`, `close`, `adj_close`, `volume`).
- Normalize `date` to `datetime64[ns]`.
- Sort chronologically and drop duplicates.
- Grouped forward-fill (`ffill()`) of missing OHLCV metrics per ticker.
- Construct a reference calendar table of valid trading dates per ticker.

In [6]:
print("=" * 80)
print("PHASE 2: CLEANING HISTORICAL PRICES")
print("=" * 80)

start_time = time.time()
prices_df = pd.read_parquet(PRICES_INPUT)
initial_price_rows = len(prices_df)
print(f"Loaded raw prices: {initial_price_rows:,} rows")

# 1. Standardize column names
prices_df.columns = [c.strip().lower().replace(" ", "_") for c in prices_df.columns]

# 2. Convert date to standard datetime64[ns] normalized (midnight)
prices_df['date'] = pd.to_datetime(prices_df['date']).dt.normalize()

# 3. Sort chronologically by stock_symbol and date
prices_df = prices_df.sort_values(by=['stock_symbol', 'date']).reset_index(drop=True)

# 4. Drop exact duplicates on (stock_symbol, date)
prices_df = prices_df.drop_duplicates(subset=['stock_symbol', 'date']).reset_index(drop=True)
print(f"Deduplication: Dropped {initial_price_rows - len(prices_df):,} duplicate rows (Remaining: {len(prices_df):,})")

# 5. Forward-fill missing OHLCV values per ticker
ohlcv_cols = ['open', 'high', 'low', 'close', 'adj_close', 'volume']
active_ohlcv = [c for c in ohlcv_cols if c in prices_df.columns]
nulls_before = prices_df[active_ohlcv].isna().sum().sum()
prices_df[active_ohlcv] = prices_df.groupby('stock_symbol')[active_ohlcv].ffill()
prices_df[active_ohlcv] = prices_df.groupby('stock_symbol')[active_ohlcv].bfill()
nulls_after = prices_df[active_ohlcv].isna().sum().sum()
print(f"Missing Values: Imputed {nulls_before - nulls_after:,} values (Remaining nulls: {nulls_after:,})")

# 6. Extract valid trading calendar per ticker
trading_calendar = prices_df[['stock_symbol', 'date']].drop_duplicates().sort_values(by=['stock_symbol', 'date']).reset_index(drop=True)

# Display calendar summary
calendar_summary = prices_df.groupby('stock_symbol').agg(
    trading_days=('date', 'count'),
    start_date=('date', lambda s: s.min().strftime('%Y-%m-%d')),
    end_date=('date', lambda s: s.max().strftime('%Y-%m-%d')),
    avg_close=('close', 'mean')
).reset_index()

print(f"\nPhase 2 completed in {time.time() - start_time:.2f}s")
display(calendar_summary)

PHASE 2: CLEANING HISTORICAL PRICES
Loaded raw prices: 67,315 rows
Deduplication: Dropped 0 duplicate rows (Remaining: 67,315)
Missing Values: Imputed 0 values (Remaining nulls: 0)

Phase 2 completed in 0.14s


,stock_symbol,trading_days,start_date,end_date,avg_close
0,AAPL,10852,1980-12-12,2023-12-28,43.776632
1,AMD,11040,1980-03-17,2023-12-28,18.178412
2,AMZN,6700,1997-05-15,2023-12-28,334.352133
3,GOOGL,3932,2004-08-19,2020-04-01,509.072183
4,JPM,11040,1980-03-17,2023-12-28,43.975880
5,MSFT,9526,1986-03-13,2023-12-28,52.903677
6,NFLX,4551,2002-05-23,2020-06-19,75.359099
7,NVDA,6275,1999-01-22,2023-12-28,42.158159
8,TSLA,3399,2010-06-29,2023-12-28,216.084819


### 📰 Step 3: Phase 3 — Clean News & US Eastern Timezone Conversion
- Drop articles with empty or missing `Article_title` or `Date`.
- Parse the raw UTC strings into timezone-aware datetime objects.
- Convert timestamps directly to **US Eastern Time (`US/Eastern`)** to align with the NYSE / NASDAQ market clock (automatically handling EST/EDT transitions).

In [7]:
print("=" * 80)
print("PHASE 3: CLEANING FINANCIAL NEWS")
print("=" * 80)

start_time = time.time()
news_df = pd.read_parquet(NEWS_INPUT)
initial_news_rows = len(news_df)
print(f"Loaded raw news rows: {initial_news_rows:,}")

# 1. Clean column headers
news_df.columns = [c.strip() for c in news_df.columns]

# 2. Drop rows where Article_title or Date is missing or whitespace
mask_valid = (
    news_df['Article_title'].notna() &
    (news_df['Article_title'].astype(str).str.strip() != "") &
    news_df['Date'].notna() &
    (news_df['Date'].astype(str).str.strip() != "")
)
news_df = news_df[mask_valid].copy()
print(f"Validation: Dropped {initial_news_rows - len(news_df):,} invalid rows (Remaining: {len(news_df):,})")

# 3. Parse Date (UTC strings) to timezone-aware datetime
print("Parsing UTC timestamps and converting to US/Eastern...")
news_df['Date_utc'] = pd.to_datetime(news_df['Date'], utc=True, errors='coerce')
news_df = news_df.dropna(subset=['Date_utc']).copy()

# Convert to US/Eastern
news_df['Date_eastern'] = news_df['Date_utc'].dt.tz_convert('US/Eastern')
news_df['Stock_symbol'] = news_df['Stock_symbol'].astype(str).str.strip().str.upper()

print(f"✓ Phase 3 completed: {len(news_df):,} clean rows in {time.time() - start_time:.2f}s")
display(news_df[['Stock_symbol', 'Date', 'Date_eastern', 'Article_title']].head(3))

PHASE 3: CLEANING FINANCIAL NEWS
Loaded raw news rows: 62,458
Validation: Dropped 0 invalid rows (Remaining: 62,458)
Parsing UTC timestamps and converting to US/Eastern...
✓ Phase 3 completed: 62,458 clean rows in 1.77s


,Stock_symbol,Date,Date_eastern,Article_title
0,AAPL,2023-12-16 22:00:00 UTC,2023-12-16 17:00:00-05:00,My 6 Largest Portfolio Holdings Heading Into 2...
1,AAPL,2023-12-16 22:00:00 UTC,2023-12-16 17:00:00-05:00,Brokers Suggest Investing in Apple (AAPL): Rea...
2,AAPL,2023-12-16 21:00:00 UTC,2023-12-16 16:00:00-05:00,"Company News for Dec 19, 2023"


### ⏱️ Step 4: Phase 4 — Vectorized 4:00 PM Cutoff & Market Calendar Alignment

#### 1. Intraday 4 PM Cutoff:
$$\text{target\_calendar\_date} = \begin{cases} \text{date}(\text{pub}), & \text{if time}(\text{pub}) \le 16:00:00 \\ \text{date}(\text{pub}) + 1\text{ day}, & \text{if time}(\text{pub}) > 16:00:00 \end{cases}$$

#### 2. Vectorized Market Session Roll Forward:
We use `pd.merge_asof(..., direction='forward')` with `by='Stock_symbol'`. For every article, it searches forward in the ticker's historical trading calendar to find the **earliest valid market trading day** $\ge \text{target\_calendar\_date}$.
- Saturday / Sunday publications $\rightarrow$ Roll to Monday (or Tuesday if Monday is a holiday).
- Friday > 4:00 PM publications $\rightarrow$ Roll to Monday.

In [8]:
print("=" * 80)
print("PHASE 4: VECTORIZED TIMESTAMP ALIGNMENT")
print("=" * 80)

start_time = time.time()

# 1. 4:00 PM Cutoff calculation
daily_cutoff = news_df['Date_eastern'].dt.normalize() + pd.Timedelta(hours=16)
is_after_4pm = news_df['Date_eastern'] > daily_cutoff

after_4pm_count = int(is_after_4pm.sum())
before_4pm_count = len(news_df) - after_4pm_count

print(f"4:00 PM Cutoff Diagnostics:")
print(f"  • Published <= 16:00:00 EST/EDT: {before_4pm_count:,d} ({before_4pm_count/len(news_df)*100:.1f}%) -> Same day target")
print(f"  • Published >  16:00:00 EST/EDT: {after_4pm_count:,d} ({after_4pm_count/len(news_df)*100:.1f}%) -> Next day target (+1 day)")

# Unaligned target date (normalized to midnight, timezone-naive)
unaligned_target = (news_df['Date_eastern'] + pd.to_timedelta(is_after_4pm.astype(int), unit='D')).dt.normalize()
news_df['lookup_date'] = unaligned_target.dt.tz_localize(None)

# 2. Vectorized forward merge against trading calendar
print("\nAligning against valid trading dates via pd.merge_asof(direction='forward')...")
cal_ref = trading_calendar.copy().rename(columns={'date': 'valid_trading_date'})
cal_ref['target_key'] = cal_ref['valid_trading_date']

# Sort by merge key as required by pd.merge_asof
news_sorted = news_df.sort_values(by='lookup_date').reset_index(drop=True)
cal_sorted = cal_ref.sort_values(by='target_key').reset_index(drop=True)

aligned_news_df = pd.merge_asof(
    news_sorted,
    cal_sorted,
    left_on='lookup_date',
    right_on='target_key',
    left_by='Stock_symbol',
    right_by='stock_symbol',
    direction='forward'
)

# Assign clean trade_date_target
aligned_news_df['trade_date_target'] = aligned_news_df['valid_trading_date']

# Shift metrics
shift_days = (aligned_news_df['trade_date_target'] - aligned_news_df['lookup_date']).dt.days
weekend_holiday_shifts = int((shift_days > 0).sum())
print(f"  • Articles rolled forward across weekends/holidays: {weekend_holiday_shifts:,d}")

# Drop any articles published past the end of price history
unmatched = int(aligned_news_df['trade_date_target'].isna().sum())
if unmatched > 0:
    print(f"  ⚠️ Dropped {unmatched:,d} articles beyond the end of price records.")
    aligned_news_df = aligned_news_df.dropna(subset=['trade_date_target']).copy()

# Drop helper columns
aligned_news_df = aligned_news_df.drop(
    columns=['lookup_date', 'target_key', 'valid_trading_date', 'stock_symbol', 'Date_utc'],
    errors='ignore'
)
aligned_news_df['Date_eastern_str'] = aligned_news_df['Date_eastern'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
aligned_news_df = aligned_news_df.sort_values(by=['trade_date_target', 'Stock_symbol']).reset_index(drop=True)

print(f"✓ Phase 4 alignment complete in {time.time() - start_time:.2f}s")

PHASE 4: VECTORIZED TIMESTAMP ALIGNMENT
4:00 PM Cutoff Diagnostics:
  • Published <= 16:00:00 EST/EDT: 146 (0.2%) -> Same day target
  • Published >  16:00:00 EST/EDT: 62,312 (99.8%) -> Next day target (+1 day)

Aligning against valid trading dates via pd.merge_asof(direction='forward')...
  • Articles rolled forward across weekends/holidays: 5,076
  ⚠️ Dropped 466 articles beyond the end of price records.
✓ Phase 4 alignment complete in 1.22s


### 💾 Step 5: Save Cleaned Datasets & Friday After 4 PM Visual Proof
- Saves `cleaned_prices.parquet` and `aligned_news.parquet` directly to Google Drive (with local `./data` backup).
- Drops timezone info from `trade_date_target` for clean Parquet export.
- Displays visual proof of 5 articles published on Friday *after* 4:00 PM EST rolling over to Monday.

In [9]:
# 1. Save cleaned datasets
target_dir = GDRIVE_DIR if os.path.exists(GDRIVE_DIR) else LOCAL_DIR
os.makedirs(target_dir, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

out_prices = os.path.join(target_dir, "cleaned_prices.parquet")
out_news = os.path.join(target_dir, "aligned_news.parquet")

print(f"Exporting cleaned data to {target_dir}...")
prices_df.to_parquet(out_prices, engine='pyarrow', compression='zstd', index=False)
aligned_news_df.to_parquet(out_news, engine='pyarrow', compression='zstd', index=False)

# Copy to local data directory if saved to Drive
if target_dir != LOCAL_DIR:
    shutil.copy2(out_prices, os.path.join(LOCAL_DIR, "cleaned_prices.parquet"))
    shutil.copy2(out_news, os.path.join(LOCAL_DIR, "aligned_news.parquet"))

print(f"  ✓ Saved {out_prices} ({os.path.getsize(out_prices)/(1024*1024):.2f} MB)")
print(f"  ✓ Saved {out_news} ({os.path.getsize(out_news)/(1024*1024):.2f} MB)")

# 2. VISUAL PROOF: Friday after 4:00 PM -> Monday target
print("\n" + "=" * 90)
print("🔍 VISUAL PROOF: FRIDAY AFTER 4:00 PM -> MONDAY TARGET SHIFT")
print("=" * 90)

friday_mask = (aligned_news_df['Date_eastern'].dt.dayofweek == 4) & (aligned_news_df['Date_eastern'].dt.hour >= 16)
proof_sample = aligned_news_df[friday_mask].head(5).copy()

proof_sample['Pub_Day'] = proof_sample['Date_eastern'].dt.day_name()
proof_sample['Published_EST'] = proof_sample['Date_eastern'].dt.strftime('%Y-%m-%d %I:%M:%S %p %Z')
proof_sample['Target_Day'] = proof_sample['trade_date_target'].dt.day_name()
proof_sample['Target_Date'] = proof_sample['trade_date_target'].dt.strftime('%Y-%m-%d')
proof_sample['Calendar_Days_Shifted'] = (
    proof_sample['trade_date_target'] - proof_sample['Date_eastern'].dt.normalize().dt.tz_localize(None)
).dt.days

proof_table = proof_sample[[
    'Stock_symbol',
    'Pub_Day',
    'Published_EST',
    'Target_Day',
    'Target_Date',
    'Calendar_Days_Shifted',
    'Article_title'
]]

display(proof_table)

Exporting cleaned data to /content/drive/MyDrive/NEXUS_Stock_AI/data...
  ✓ Saved /content/drive/MyDrive/NEXUS_Stock_AI/data/cleaned_prices.parquet (1.79 MB)
  ✓ Saved /content/drive/MyDrive/NEXUS_Stock_AI/data/aligned_news.parquet (91.84 MB)

🔍 VISUAL PROOF: FRIDAY AFTER 4:00 PM -> MONDAY TARGET SHIFT


,Stock_symbol,Pub_Day,Published_EST,Target_Day,Target_Date,Calendar_Days_Shifted,Article_title
36,NVDA,Friday,2011-04-15 08:00:00 PM EDT,Monday,2011-04-18,3,"CEOWORLD Daily Business Roundup- JNJ, CHK, AMD..."
440,NVDA,Friday,2013-03-01 07:00:00 PM EST,Monday,2013-03-04,3,Short Interest in Chip Makers on the Rise (AMA...
534,NVDA,Friday,2013-10-25 08:00:00 PM EDT,Monday,2013-10-28,3,Short Sellers Pile On Applied Materials and ST...
635,NVDA,Friday,2014-08-08 08:00:00 PM EDT,Monday,2014-08-11,3,CNBC's Stock Pops & Drops From August 8
743,NVDA,Friday,2015-06-19 08:00:00 PM EDT,Monday,2015-06-22,3,Weekly Tech Highlights: Sony Beat Apple To The...
